In [5]:
import pandas as pd

In [6]:
import os

print(os.getcwd())
print(os.listdir())


/content
['.config', 'MediRent-Ai', 'sample_data']


In [7]:
!git clone https://github.com/Atha1407/MediRent-Ai.git

fatal: destination path 'MediRent-Ai' already exists and is not an empty directory.


In [8]:
!ls


MediRent-Ai  sample_data


In [9]:
!ls MediRent-Ai/data/processed

cleaned_medical_equipment_dataset.csv
demand_prediction_dataset.csv
demand_prediction_model_dataset.csv
feature_engineered_medical_equipment_dataset.csv


In [10]:
import pandas as pd

demand_model_df = pd.read_csv(
    "/content/MediRent-Ai/data/processed/demand_prediction_dataset.csv"
)

print("Shape:", demand_model_df.shape)

Shape: (496, 11)


In [11]:
print(demand_model_df.columns.tolist())

['Equipment', 'Equipment_Category', 'Year', 'Month_Number', 'Quarter', 'Historical_Demand_Lag_1', 'Historical_Demand_Lag_2', 'Historical_Demand_Rolling_3M', 'Historical_Demand_Rolling_6M', 'Historical_Demand_Trend_2M', 'Monthly_Demand']


In [12]:
demand_model_df = demand_model_df.sort_values(
    ["Year", "Month_Number"]
).reset_index(drop=True)

print(
    demand_model_df[
        ["Year", "Month_Number"]
    ].drop_duplicates().head()
)

print("\nLast 5 months:")
print(
    demand_model_df[
        ["Year", "Month_Number"]
    ].drop_duplicates().tail()
)

    Year  Month_Number
0   2024             1
16  2024             2
32  2024             3
48  2024             4
64  2024             5

Last 5 months:
     Year  Month_Number
416  2026             3
432  2026             4
448  2026             5
464  2026             6
480  2026             7


In [13]:
train_df = demand_model_df[
    demand_model_df["Year"] < 2026
].copy()

test_df = demand_model_df[
    demand_model_df["Year"] == 2026
].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTraining period:")
print(
    train_df[["Year", "Month_Number"]]
    .drop_duplicates()
    .iloc[[0, -1]]
)

print("\nTesting period:")
print(
    test_df[["Year", "Month_Number"]]
    .drop_duplicates()
    .iloc[[0, -1]]
)

Train shape: (384, 11)
Test shape: (112, 11)

Training period:
     Year  Month_Number
0    2024             1
368  2025            12

Testing period:
     Year  Month_Number
384  2026             1
480  2026             7


In [14]:
X_train = train_df.drop("Monthly_Demand", axis=1)
y_train = train_df["Monthly_Demand"]

X_test = test_df.drop("Monthly_Demand", axis=1)
y_test = test_df["Monthly_Demand"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (384, 10)
y_train: (384,)
X_test: (112, 10)
y_test: (112,)


In [15]:
print(X_train.dtypes)

Equipment                        object
Equipment_Category               object
Year                              int64
Month_Number                      int64
Quarter                           int64
Historical_Demand_Lag_1         float64
Historical_Demand_Lag_2         float64
Historical_Demand_Rolling_3M    float64
Historical_Demand_Rolling_6M    float64
Historical_Demand_Trend_2M      float64
dtype: object


In [16]:
print("Training missing values:")
print(X_train.isnull().sum())

print("\nTesting missing values:")
print(X_test.isnull().sum())

Training missing values:
Equipment                        0
Equipment_Category               0
Year                             0
Month_Number                     0
Quarter                          0
Historical_Demand_Lag_1         16
Historical_Demand_Lag_2         32
Historical_Demand_Rolling_3M    16
Historical_Demand_Rolling_6M    16
Historical_Demand_Trend_2M      32
dtype: int64

Testing missing values:
Equipment                       0
Equipment_Category              0
Year                            0
Month_Number                    0
Quarter                         0
Historical_Demand_Lag_1         0
Historical_Demand_Lag_2         0
Historical_Demand_Rolling_3M    0
Historical_Demand_Rolling_6M    0
Historical_Demand_Trend_2M      0
dtype: int64


In [17]:
missing_rows = X_train[
    X_train[
        [
            "Historical_Demand_Lag_1",
            "Historical_Demand_Lag_2",
            "Historical_Demand_Rolling_3M",
            "Historical_Demand_Rolling_6M",
            "Historical_Demand_Trend_2M"
        ]
    ].isnull().any(axis=1)
]

print(missing_rows[
    [
        "Equipment",
        "Year",
        "Month_Number",
        "Historical_Demand_Lag_1",
        "Historical_Demand_Lag_2",
        "Historical_Demand_Rolling_3M",
        "Historical_Demand_Rolling_6M",
        "Historical_Demand_Trend_2M"
    ]
].head(20))

              Equipment  Year  Month_Number  Historical_Demand_Lag_1  \
0         BiPAP Machine  2024             1                      NaN   
1          CPAP Machine  2024             1                      NaN   
2         Commode Chair  2024             1                      NaN   
3           ECG Machine  2024             1                      NaN   
4            Fowler Bed  2024             1                      NaN   
5          Hospital Bed  2024             1                      NaN   
6         Infusion Pump  2024             1                      NaN   
7             Nebulizer  2024             1                      NaN   
8   Oxygen Concentrator  2024             1                      NaN   
9       Patient Monitor  2024             1                      NaN   
10      Suction Machine  2024             1                      NaN   
11   Surgical Headlight  2024             1                      NaN   
12       Surgical Loupe  2024             1                     

In [18]:
baseline_pred = X_test["Historical_Demand_Lag_1"]

print("Baseline predictions:")
print(baseline_pred.head(10))

print("\nActual demand:")
print(y_test.head(10))

Baseline predictions:
384     3.0
385     9.0
386     5.0
387     6.0
388     7.0
389     9.0
390     1.0
391     5.0
392    13.0
393     6.0
Name: Historical_Demand_Lag_1, dtype: float64

Actual demand:
384     2
385     6
386     5
387     4
388     6
389    18
390     3
391     4
392     5
393     4
Name: Monthly_Demand, dtype: int64


In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

baseline_pred = X_test["Historical_Demand_Lag_1"]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)

Baseline MAE: 4.589285714285714
Baseline RMSE: 5.885818306986476


In [20]:
print("Equipment types:", X_train["Equipment"].nunique())
print("Equipment categories:", X_train["Equipment_Category"].nunique())

print("\nEquipment types:")
print(X_train["Equipment"].unique())

print("\nEquipment categories:")
print(X_train["Equipment_Category"].unique())

Equipment types: 16
Equipment categories: 7

Equipment types:
['BiPAP Machine' 'CPAP Machine' 'Commode Chair' 'ECG Machine' 'Fowler Bed'
 'Hospital Bed' 'Infusion Pump' 'Nebulizer' 'Oxygen Concentrator'
 'Patient Monitor' 'Suction Machine' 'Surgical Headlight' 'Surgical Loupe'
 'Ventilator' 'Walker' 'Wheelchair']

Equipment categories:
['Respiratory Equipment' 'Mobility Equipment' 'Diagnostic Equipment'
 'Patient Care Equipment' 'Critical Care Equipment' 'Monitoring Equipment'
 'Surgical Equipment']


In [21]:
numeric_features = [
    "Year",
    "Month_Number",
    "Quarter",
    "Historical_Demand_Lag_1",
    "Historical_Demand_Lag_2",
    "Historical_Demand_Rolling_3M",
    "Historical_Demand_Rolling_6M",
    "Historical_Demand_Trend_2M"
]

print(X_train[numeric_features].describe().T)

                              count         mean       std          min  \
Year                          384.0  2024.500000  0.500652  2024.000000   
Month_Number                  384.0     6.500000  3.456556     1.000000   
Quarter                       384.0     2.500000  1.119493     1.000000   
Historical_Demand_Lag_1       368.0     5.548913  3.953793     0.000000   
Historical_Demand_Lag_2       352.0     5.607955  3.997826     0.000000   
Historical_Demand_Rolling_3M  368.0     5.561141  3.413849     0.333333   
Historical_Demand_Rolling_6M  368.0     5.544656  3.281459     0.333333   
Historical_Demand_Trend_2M    352.0    -0.005682  3.618561   -11.000000   

                                      25%          50%          75%     max  
Year                          2024.000000  2024.500000  2025.000000  2025.0  
Month_Number                     3.750000     6.500000     9.250000    12.0  
Quarter                          1.750000     2.500000     3.250000     4.0  
Historical_D

In [22]:
categorical_features = [
    "Equipment",
    "Equipment_Category"
]

numeric_features = [
    "Year",
    "Month_Number",
    "Quarter",
    "Historical_Demand_Lag_1",
    "Historical_Demand_Lag_2",
    "Historical_Demand_Rolling_3M",
    "Historical_Demand_Rolling_6M",
    "Historical_Demand_Trend_2M"
]

print("Categorical features:", categorical_features)
print("Number of categorical features:", len(categorical_features))

print("\nNumeric features:", numeric_features)
print("Number of numeric features:", len(numeric_features))

Categorical features: ['Equipment', 'Equipment_Category']
Number of categorical features: 2

Numeric features: ['Year', 'Month_Number', 'Quarter', 'Historical_Demand_Lag_1', 'Historical_Demand_Lag_2', 'Historical_Demand_Rolling_3M', 'Historical_Demand_Rolling_6M', 'Historical_Demand_Trend_2M']
Number of numeric features: 8


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")


Preprocessing pipeline created successfully.


In [24]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape:", X_test_processed.shape)

Training processed shape: (384, 31)
Testing processed shape: (112, 31)


In [25]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [26]:
rf_pred = rf_model.predict(X_test_processed)

print("First 10 Random Forest predictions:")
print(rf_pred[:10])

print("\nFirst 10 actual values:")
print(y_test.head(10).values)

First 10 Random Forest predictions:
[6.87  8.415 6.515 5.06  6.285 9.425 2.12  5.125 8.215 5.73 ]

First 10 actual values:
[ 2  6  5  4  6 18  3  4  5  4]


In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Results")
print("---------------------")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R²  :", rf_r2)

Random Forest Results
---------------------
MAE : 5.862410714285715
RMSE: 8.113673341297746
R²  : -0.027085012238562634


In [28]:
import xgboost

print("XGBoost version:", xgboost.__version__)

XGBoost version: 3.4.1


In [29]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train_processed, y_train)

print("XGBoost trained successfully.")

XGBoost trained successfully.


In [30]:
xgb_pred = xgb_model.predict(X_test_processed)

print("First 10 XGBoost predictions:")
print(xgb_pred[:10])

print("\nFirst 10 actual values:")
print(y_test.head(10).values)

First 10 XGBoost predictions:
[ 5.5189786  8.602543   4.9775834  4.9204106  6.0739055 11.364992
  2.3808823  5.3652434  7.8038287  4.9775834]

First 10 actual values:
[ 2  6  5  4  6 18  3  4  5  4]


In [31]:
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost Results")
print("----------------")
print("MAE :", xgb_mae)
print("RMSE:", xgb_rmse)
print("R²  :", xgb_r2)

XGBoost Results
----------------
MAE : 6.4366865158081055
RMSE: 8.846514482068365
R²  : -0.2210003137588501


In [32]:
monthly_demand = (
    demand_model_df
    .groupby(["Year", "Month_Number"])["Monthly_Demand"]
    .sum()
    .reset_index()
)

print(monthly_demand)

    Year  Month_Number  Monthly_Demand
0   2024             1              70
1   2024             2              93
2   2024             3              72
3   2024             4             101
4   2024             5             112
5   2024             6              93
6   2024             7             102
7   2024             8             102
8   2024             9              83
9   2024            10              87
10  2024            11              75
11  2024            12              76
12  2025             1              96
13  2025             2              76
14  2025             3             102
15  2025             4              97
16  2025             5              94
17  2025             6              89
18  2025             7             102
19  2025             8              92
20  2025             9              88
21  2025            10              72
22  2025            11              68
23  2025            12             102
24  2026             1   

In [33]:
equipment_year_demand = (
    demand_model_df
    .groupby(["Equipment", "Year"])["Monthly_Demand"]
    .sum()
    .unstack()
)

print(equipment_year_demand)

Year                 2024  2025  2026
Equipment                            
BiPAP Machine          61    63    76
CPAP Machine           93   101   107
Commode Chair          61    53    72
ECG Machine            45    59    55
Fowler Bed             82    88   115
Hospital Bed          142   123   188
Infusion Pump          25    43    53
Nebulizer              65    59    70
Oxygen Concentrator   116   102   129
Patient Monitor        55    57    80
Suction Machine        53    67    93
Surgical Headlight     29    37    46
Surgical Loupe         25    19    32
Ventilator             26    25    31
Walker                 37    32    35
Wheelchair            151   150   174


In [34]:
train_df[[
    "Historical_Demand_Lag_1",
    "Historical_Demand_Lag_2",
    "Historical_Demand_Rolling_3M",
    "Historical_Demand_Rolling_6M",
    "Monthly_Demand"
]].describe()

,Historical_Demand_Lag_1,Historical_Demand_Lag_2,Historical_Demand_Rolling_3M,Historical_Demand_Rolling_6M,Monthly_Demand
count,368.000000,352.000000,368.000000,368.000000,384.000000
mean,5.548913,5.607955,5.561141,5.544656,5.583333
std,3.953793,3.997826,3.413849,3.281459,3.995864
min,0.000000,0.000000,0.333333,0.333333,0.000000
25%,3.000000,3.000000,3.000000,3.166667,3.000000
50%,5.000000,5.000000,5.000000,4.833333,5.000000
75%,8.000000,8.000000,7.333333,7.541667,8.000000
max,21.000000,21.000000,17.000000,15.000000,21.000000


In [35]:
monthly_pattern = (
    train_df
    .groupby("Month_Number")["Monthly_Demand"]
    .mean()
    .round(2)
)

print(monthly_pattern)

Month_Number
1     5.19
2     5.28
3     5.44
4     6.19
5     6.44
6     5.69
7     6.38
8     6.06
9     5.34
10    4.97
11    4.47
12    5.56
Name: Monthly_Demand, dtype: float64


In [36]:
category_year_demand = (
    demand_model_df
    .groupby(["Equipment_Category", "Year"])["Monthly_Demand"]
    .sum()
    .unstack()
)

print(category_year_demand)

Year                     2024  2025  2026
Equipment_Category                       
Critical Care Equipment    51    68    84
Diagnostic Equipment       45    59    55
Mobility Equipment        249   235   281
Monitoring Equipment       55    57    80
Patient Care Equipment    224   211   303
Respiratory Equipment     335   325   382
Surgical Equipment        107   123   171


In [37]:
year_month_demand = (
    demand_model_df
    .groupby(["Year", "Month_Number"])["Monthly_Demand"]
    .sum()
    .reset_index()
)

print(year_month_demand)

    Year  Month_Number  Monthly_Demand
0   2024             1              70
1   2024             2              93
2   2024             3              72
3   2024             4             101
4   2024             5             112
5   2024             6              93
6   2024             7             102
7   2024             8             102
8   2024             9              83
9   2024            10              87
10  2024            11              75
11  2024            12              76
12  2025             1              96
13  2025             2              76
14  2025             3             102
15  2025             4              97
16  2025             5              94
17  2025             6              89
18  2025             7             102
19  2025             8              92
20  2025             9              88
21  2025            10              72
22  2025            11              68
23  2025            12             102
24  2026             1   

In [38]:
master_df["Rental_Start_Date"] = pd.to_datetime(master_df["Rental_Start_Date"])

monthly_records = (
    master_df
    .groupby(master_df["Rental_Start_Date"].dt.to_period("M"))
    .size()
)

print(monthly_records)

NameError: name 'master_df' is not defined

In [39]:
business_monthly_demand = (
    demand_model_df
    .groupby(["Year", "Month_Number"])["Monthly_Demand"]
    .sum()
    .reset_index()
    .rename(columns={"Monthly_Demand": "Total_Business_Demand"})
)

business_monthly_demand["Previous_Month_Business_Demand"] = (
    business_monthly_demand["Total_Business_Demand"].shift(1)
)

print(business_monthly_demand)

    Year  Month_Number  Total_Business_Demand  Previous_Month_Business_Demand
0   2024             1                     70                             NaN
1   2024             2                     93                            70.0
2   2024             3                     72                            93.0
3   2024             4                    101                            72.0
4   2024             5                    112                           101.0
5   2024             6                     93                           112.0
6   2024             7                    102                            93.0
7   2024             8                    102                           102.0
8   2024             9                     83                           102.0
9   2024            10                     87                            83.0
10  2024            11                     75                            87.0
11  2024            12                     76                   

In [40]:
demand_model_df = demand_model_df.merge(
    business_monthly_demand[
        ["Year", "Month_Number", "Previous_Month_Business_Demand"]
    ],
    on=["Year", "Month_Number"],
    how="left"
)

print(demand_model_df.columns.tolist())
print(demand_model_df[
    ["Year", "Month_Number", "Equipment", "Monthly_Demand",
     "Previous_Month_Business_Demand"]
].head(20))

['Equipment', 'Equipment_Category', 'Year', 'Month_Number', 'Quarter', 'Historical_Demand_Lag_1', 'Historical_Demand_Lag_2', 'Historical_Demand_Rolling_3M', 'Historical_Demand_Rolling_6M', 'Historical_Demand_Trend_2M', 'Monthly_Demand', 'Previous_Month_Business_Demand']
    Year  Month_Number            Equipment  Monthly_Demand  \
0   2024             1        BiPAP Machine               7   
1   2024             1         CPAP Machine               7   
2   2024             1        Commode Chair              10   
3   2024             1          ECG Machine               3   
4   2024             1           Fowler Bed               1   
5   2024             1         Hospital Bed               6   
6   2024             1        Infusion Pump               1   
7   2024             1            Nebulizer               3   
8   2024             1  Oxygen Concentrator               8   
9   2024             1      Patient Monitor               4   
10  2024             1      Suction 

In [41]:
numeric_features = [
    "Year",
    "Month_Number",
    "Quarter",
    "Historical_Demand_Lag_1",
    "Historical_Demand_Lag_2",
    "Historical_Demand_Rolling_3M",
    "Historical_Demand_Rolling_6M",
    "Historical_Demand_Trend_2M",
    "Previous_Month_Business_Demand"
]

print(numeric_features)

['Year', 'Month_Number', 'Quarter', 'Historical_Demand_Lag_1', 'Historical_Demand_Lag_2', 'Historical_Demand_Rolling_3M', 'Historical_Demand_Rolling_6M', 'Historical_Demand_Trend_2M', 'Previous_Month_Business_Demand']


In [42]:
train_df = demand_model_df[
    demand_model_df["Year"] < 2026
].copy()

test_df = demand_model_df[
    demand_model_df["Year"] == 2026
].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nNew feature check:")
print(train_df["Previous_Month_Business_Demand"].head())
print(test_df["Previous_Month_Business_Demand"].head())

Train shape: (384, 12)
Test shape: (112, 12)

New feature check:
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: Previous_Month_Business_Demand, dtype: float64
384    102.0
385    102.0
386    102.0
387    102.0
388    102.0
Name: Previous_Month_Business_Demand, dtype: float64


In [43]:
X_train = train_df.drop("Monthly_Demand", axis=1)
y_train = train_df["Monthly_Demand"]

X_test = test_df.drop("Monthly_Demand", axis=1)
y_test = test_df["Monthly_Demand"]

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (384, 31)
Testing shape: (112, 31)


In [44]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rf_model_new = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model_new.fit(X_train_processed, y_train)

rf_pred_new = rf_model_new.predict(X_test_processed)

rf_mae_new = mean_absolute_error(y_test, rf_pred_new)
rf_rmse_new = np.sqrt(mean_squared_error(y_test, rf_pred_new))
rf_r2_new = r2_score(y_test, rf_pred_new)

print("Random Forest with Business Lag")
print("MAE :", rf_mae_new)
print("RMSE:", rf_rmse_new)
print("R²  :", rf_r2_new)

Random Forest with Business Lag
MAE : 5.862410714285715
RMSE: 8.113673341297746
R²  : -0.027085012238562634


In [45]:
from xgboost import XGBRegressor

xgb_model_new = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model_new.fit(X_train_processed, y_train)

xgb_pred_new = xgb_model_new.predict(X_test_processed)

xgb_mae_new = mean_absolute_error(y_test, xgb_pred_new)
xgb_rmse_new = np.sqrt(mean_squared_error(y_test, xgb_pred_new))
xgb_r2_new = r2_score(y_test, xgb_pred_new)

print("XGBoost with Business Lag")
print("MAE :", xgb_mae_new)
print("RMSE:", xgb_rmse_new)
print("R²  :", xgb_r2_new)

XGBoost with Business Lag
MAE : 6.4366865158081055
RMSE: 8.846514482068365
R²  : -0.2210003137588501


In [46]:
correlation = train_df[
    [
        "Historical_Demand_Lag_1",
        "Historical_Demand_Lag_2",
        "Historical_Demand_Rolling_3M",
        "Historical_Demand_Rolling_6M",
        "Historical_Demand_Trend_2M",
        "Previous_Month_Business_Demand",
        "Monthly_Demand"
    ]
].corr()["Monthly_Demand"].sort_values(ascending=False)

print(correlation)

Monthly_Demand                    1.000000
Historical_Demand_Rolling_6M      0.696594
Historical_Demand_Rolling_3M      0.679383
Historical_Demand_Lag_1           0.581725
Historical_Demand_Lag_2           0.574069
Previous_Month_Business_Demand    0.017646
Historical_Demand_Trend_2M        0.009338
Name: Monthly_Demand, dtype: float64


In [47]:
selected_features = [
    "Equipment",
    "Equipment_Category",
    "Historical_Demand_Lag_1",
    "Historical_Demand_Lag_2",
    "Historical_Demand_Rolling_3M",
    "Historical_Demand_Rolling_6M"
]

X_train_selected = train_df[selected_features]
X_test_selected = test_df[selected_features]

selected_numeric = [
    "Historical_Demand_Lag_1",
    "Historical_Demand_Lag_2",
    "Historical_Demand_Rolling_3M",
    "Historical_Demand_Rolling_6M"
]

selected_categorical = [
    "Equipment",
    "Equipment_Category"
]

selected_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            selected_numeric
        ),
        (
            "cat",
            Pipeline([
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            selected_categorical
        )
    ]
)

X_train_selected_processed = selected_preprocessor.fit_transform(
    X_train_selected
)

X_test_selected_processed = selected_preprocessor.transform(
    X_test_selected
)

print("Training shape:", X_train_selected_processed.shape)
print("Testing shape:", X_test_selected_processed.shape)

Training shape: (384, 27)
Testing shape: (112, 27)


In [48]:
rf_selected = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_selected.fit(
    X_train_selected_processed,
    y_train
)

rf_selected_pred = rf_selected.predict(
    X_test_selected_processed
)

rf_selected_mae = mean_absolute_error(
    y_test, rf_selected_pred
)

rf_selected_rmse = np.sqrt(
    mean_squared_error(y_test, rf_selected_pred)
)

rf_selected_r2 = r2_score(
    y_test, rf_selected_pred
)

print("Random Forest - Historical Features Only")
print("MAE :", rf_selected_mae)
print("RMSE:", rf_selected_rmse)
print("R²  :", rf_selected_r2)

Random Forest - Historical Features Only
MAE : 6.273333333333333
RMSE: 8.706579739137997
R²  : -0.18267799866890466


In [49]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=10.0)

ridge_model.fit(
    X_train_processed,
    y_train
)

ridge_pred = ridge_model.predict(
    X_test_processed
)

ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)

print("Ridge Regression")
print("MAE :", ridge_mae)
print("RMSE:", ridge_rmse)
print("R²  :", ridge_r2)

Ridge Regression
MAE : 5.468112020601792
RMSE: 7.231025517822611
R²  : 0.1842235905575812


In [50]:
demand_change = (
    train_df["Monthly_Demand"]
    - train_df["Historical_Demand_Lag_1"]
)

print(demand_change.describe())

count    368.000000
mean       0.086957
std        3.647707
min      -11.000000
25%       -2.000000
50%        0.000000
75%        2.000000
max       11.000000
dtype: float64


In [51]:
change_train_mask = train_df["Historical_Demand_Lag_1"].notna()

X_change_train = train_df.loc[
    change_train_mask,
    selected_features
]

y_change_train = (
    train_df.loc[change_train_mask, "Monthly_Demand"]
    - train_df.loc[change_train_mask, "Historical_Demand_Lag_1"]
)

print("Training rows:", len(X_change_train))
print("Target mean:", y_change_train.mean())
print("Target std:", y_change_train.std())

Training rows: 368
Target mean: 0.08695652173913043
Target std: 3.647707425026704


In [52]:
change_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            selected_numeric
        ),
        (
            "cat",
            Pipeline([
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            selected_categorical
        )
    ]
)

X_change_train_processed = change_preprocessor.fit_transform(
    X_change_train
)

ridge_change = Ridge(alpha=10.0)

ridge_change.fit(
    X_change_train_processed,
    y_change_train
)

# Prepare test data
X_change_test = test_df[selected_features]

X_change_test_processed = change_preprocessor.transform(
    X_change_test
)

predicted_change = ridge_change.predict(
    X_change_test_processed
)

# Reconstruct predicted demand
change_ridge_pred = (
    test_df["Historical_Demand_Lag_1"].values
    + predicted_change
)

print("First 10 predicted demand:", change_ridge_pred[:10])
print("First 10 actual demand:", y_test.values[:10])

First 10 predicted demand: [4.88466443 8.20404259 4.49588888 4.88040371 7.1530261  9.51688333
 3.09693719 5.18958155 8.91163232 4.64093712]
First 10 actual demand: [ 2  6  5  4  6 18  3  4  5  4]


In [53]:
change_ridge_mae = mean_absolute_error(
    y_test, change_ridge_pred
)

change_ridge_rmse = np.sqrt(
    mean_squared_error(y_test, change_ridge_pred)
)

change_ridge_r2 = r2_score(
    y_test, change_ridge_pred
)

print("Ridge - Demand Change Prediction")
print("MAE :", change_ridge_mae)
print("RMSE:", change_ridge_rmse)
print("R²  :", change_ridge_r2)

Ridge - Demand Change Prediction
MAE : 5.491106574857328
RMSE: 7.276869262811939
R²  : 0.173846972315838


In [54]:
comparison_df = pd.DataFrame({
    "Year": test_df["Year"].values,
    "Month": test_df["Month_Number"].values,
    "Equipment": test_df["Equipment"].values,
    "Actual": y_test.values,
    "Baseline": baseline_pred.values,
    "Ridge_Change": change_ridge_pred
})

comparison_df["Baseline_Error"] = (
    comparison_df["Actual"] - comparison_df["Baseline"]
).abs()

comparison_df["Ridge_Change_Error"] = (
    comparison_df["Actual"] - comparison_df["Ridge_Change"]
).abs()

print(
    comparison_df[
        ["Year", "Month", "Actual", "Baseline", "Ridge_Change",
         "Baseline_Error", "Ridge_Change_Error"]
    ].head(20)
)

    Year  Month  Actual  Baseline  Ridge_Change  Baseline_Error  \
0   2026      1       2       3.0      4.884664             1.0   
1   2026      1       6       9.0      8.204043             3.0   
2   2026      1       5       5.0      4.495889             0.0   
3   2026      1       4       6.0      4.880404             2.0   
4   2026      1       6       7.0      7.153026             1.0   
5   2026      1      18       9.0      9.516883             9.0   
6   2026      1       3       1.0      3.096937             2.0   
7   2026      1       4       5.0      5.189582             1.0   
8   2026      1       5      13.0      8.911632             8.0   
9   2026      1       4       6.0      4.640937             2.0   
10  2026      1       7       9.0      5.097396             2.0   
11  2026      1       7       1.0      3.199899             6.0   
12  2026      1       6       2.0      1.972211             4.0   
13  2026      1       3       4.0      2.514443             1.

In [55]:
monthly_errors = (
    comparison_df
    .groupby("Month")
    .agg(
        Baseline_MAE=("Baseline_Error", "mean"),
        Ridge_Change_MAE=("Ridge_Change_Error", "mean")
    )
    .round(3)
)

print(monthly_errors)

       Baseline_MAE  Ridge_Change_MAE
Month                                
1             3.250             2.185
2             6.875             6.407
3             5.562             7.856
4             3.438             5.719
5             3.438             6.041
6             4.438             4.267
7             5.125             5.964


In [56]:
final_results = pd.DataFrame({
    "Model": [
        "Previous-Month Baseline",
        "Random Forest",
        "XGBoost",
        "Random Forest + Business Lag",
        "Ridge Regression",
        "Ridge - Demand Change"
    ],
    "MAE": [
        4.589285714285714,
        5.862410714285715,
        6.4366865158081055,
        5.670624999999999,
        5.96067684678338,
        5.491106574857328
    ],
    "RMSE": [
        5.885818306986476,
        8.113673341297746,
        8.846514482068365,
        7.935358175596108,
        7.7006721197656836,
        7.276869262811939
    ],
    "R2": [
        np.nan,
        -0.027085012238562634,
        -0.2210003137588501,
        0.017563651469622354,
        0.07481492389462852,
        0.173846972315838
    ]
})

print(final_results.round(4))

                          Model     MAE    RMSE      R2
0       Previous-Month Baseline  4.5893  5.8858     NaN
1                 Random Forest  5.8624  8.1137 -0.0271
2                       XGBoost  6.4367  8.8465 -0.2210
3  Random Forest + Business Lag  5.6706  7.9354  0.0176
4              Ridge Regression  5.9607  7.7007  0.0748
5         Ridge - Demand Change  5.4911  7.2769  0.1738


In [57]:
print("Demand dataset shape:", demand_model_df.shape)
print("\nColumns:")
print(demand_model_df.columns.tolist())

Demand dataset shape: (496, 12)

Columns:
['Equipment', 'Equipment_Category', 'Year', 'Month_Number', 'Quarter', 'Historical_Demand_Lag_1', 'Historical_Demand_Lag_2', 'Historical_Demand_Rolling_3M', 'Historical_Demand_Rolling_6M', 'Historical_Demand_Trend_2M', 'Monthly_Demand', 'Previous_Month_Business_Demand']


In [58]:
print("Missing values:")
print(demand_model_df.isnull().sum())

print("\nDuplicate rows:", demand_model_df.duplicated().sum())

print("\nTarget summary:")
print(demand_model_df["Monthly_Demand"].describe())

Missing values:
Equipment                          0
Equipment_Category                 0
Year                               0
Month_Number                       0
Quarter                            0
Historical_Demand_Lag_1           16
Historical_Demand_Lag_2           32
Historical_Demand_Rolling_3M      16
Historical_Demand_Rolling_6M      16
Historical_Demand_Trend_2M        32
Monthly_Demand                     0
Previous_Month_Business_Demand    16
dtype: int64

Duplicate rows: 0

Target summary:
count    496.000000
mean       7.056452
std        5.857641
min        0.000000
25%        3.000000
50%        6.000000
75%        9.000000
max       40.000000
Name: Monthly_Demand, dtype: float64


In [59]:
demand_model_df.to_csv(
    "/content/MediRent-Ai/data/processed/demand_prediction_model_dataset.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", demand_model_df.shape)

Saved successfully.
Shape: (496, 12)


In [60]:
import os

model_path = "/content/MediRent-Ai/data/processed/demand_prediction_model_dataset.csv"

print("File exists:", os.path.exists(model_path))
print("File size:", round(os.path.getsize(model_path) / 1024, 2), "KB")

File exists: True
File size: 42.96 KB


In [ ]:
%cd /content/MediRent-Ai

/content/MediRent-Ai


In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/demand_prediction_model_dataset.csv

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
!git add data/processed/demand_prediction_model_dataset.csv

In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   data/processed/demand_prediction_model_dataset.csv



In [ ]:
!git commit -m "Add demand prediction modeling dataset"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@3eba85a291d6.(none)')


In [ ]:
!git config --global user.name "Atharva Ingle"
!git config --global user.email "atharvaingle40@gmail.com"

In [ ]:
!git config --global user.name
!git config --global user.email

Atharva Ingle
atharvaingle40@gmail.com


In [ ]:
!git commit -m "Add demand prediction modeling dataset"

[main c9ff934] Add demand prediction modeling dataset
 1 file changed, 497 insertions(+)
 create mode 100644 data/processed/demand_prediction_model_dataset.csv


In [ ]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
!git remote -v

origin	https://github.com/Atha1407/MediRent-Ai.git (fetch)
origin	https://github.com/Atha1407/MediRent-Ai.git (push)


In [ ]:
!git config --global credential.helper store

In [ ]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
!git config --global credential.helper store

In [ ]:
!git push origin main

Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 6.90 KiB | 2.30 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Atha1407/MediRent-Ai.git
   f6e3246..c9ff934  main -> main


In [62]:
import pandas as pd

model_path = "/content/MediRent-Ai/data/processed/demand_prediction_model_dataset.csv"

demand_model_df = pd.read_csv(model_path)

print("Demand modeling dataset shape:", demand_model_df.shape)
print("Columns:")
print(demand_model_df.columns.tolist())

Demand modeling dataset shape: (496, 12)
Columns:
['Equipment', 'Equipment_Category', 'Year', 'Month_Number', 'Quarter', 'Historical_Demand_Lag_1', 'Historical_Demand_Lag_2', 'Historical_Demand_Rolling_3M', 'Historical_Demand_Rolling_6M', 'Historical_Demand_Trend_2M', 'Monthly_Demand', 'Previous_Month_Business_Demand']


In [63]:
# Create time-aware train/test split

train_df = demand_model_df[
    (demand_model_df["Year"] < 2026)
].copy()

test_df = demand_model_df[
    (demand_model_df["Year"] == 2026)
].copy()

print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

print("\nTraining period:")
print(train_df[["Year", "Month_Number"]].drop_duplicates().sort_values(
    ["Year", "Month_Number"]
).head(1))
print("to")
print(train_df[["Year", "Month_Number"]].drop_duplicates().sort_values(
    ["Year", "Month_Number"]
).tail(1))

print("\nTesting period:")
print(test_df[["Year", "Month_Number"]].drop_duplicates().sort_values(
    ["Year", "Month_Number"]
).head(1))
print("to")
print(test_df[["Year", "Month_Number"]].drop_duplicates().sort_values(
    ["Year", "Month_Number"]
).tail(1))

Training set shape: (384, 12)
Testing set shape: (112, 12)

Training period:
   Year  Month_Number
0  2024             1
to
     Year  Month_Number
368  2025            12

Testing period:
     Year  Month_Number
384  2026             1
to
     Year  Month_Number
480  2026             7


In [64]:
# Define target and input features

target = "Monthly_Demand"

feature_columns = [
    "Equipment",
    "Equipment_Category",
    "Year",
    "Month_Number",
    "Quarter",
    "Historical_Demand_Lag_1",
    "Historical_Demand_Lag_2",
    "Historical_Demand_Rolling_3M",
    "Historical_Demand_Rolling_6M",
    "Historical_Demand_Trend_2M",
    "Previous_Month_Business_Demand"
]

X_train = train_df[feature_columns].copy()
y_train = train_df[target].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target].copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (384, 11)
y_train shape: (384,)
X_test shape: (112, 11)
y_test shape: (112,)


In [65]:
print("Missing values in training features:")
print(X_train.isnull().sum())

print("\nMissing values in testing features:")
print(X_test.isnull().sum())

Missing values in training features:
Equipment                          0
Equipment_Category                 0
Year                               0
Month_Number                       0
Quarter                            0
Historical_Demand_Lag_1           16
Historical_Demand_Lag_2           32
Historical_Demand_Rolling_3M      16
Historical_Demand_Rolling_6M      16
Historical_Demand_Trend_2M        32
Previous_Month_Business_Demand    16
dtype: int64

Missing values in testing features:
Equipment                         0
Equipment_Category                0
Year                              0
Month_Number                      0
Quarter                           0
Historical_Demand_Lag_1           0
Historical_Demand_Lag_2           0
Historical_Demand_Rolling_3M      0
Historical_Demand_Rolling_6M      0
Historical_Demand_Trend_2M        0
Previous_Month_Business_Demand    0
dtype: int64


In [66]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = [
    "Equipment",
    "Equipment_Category"
]

numerical_features = [
    "Year",
    "Month_Number",
    "Quarter",
    "Historical_Demand_Lag_1",
    "Historical_Demand_Lag_2",
    "Historical_Demand_Rolling_3M",
    "Historical_Demand_Rolling_6M",
    "Historical_Demand_Trend_2M",
    "Previous_Month_Business_Demand"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numerical_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (384, 32)
Processed testing shape: (112, 32)


In [67]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

baseline_predictions = X_test["Historical_Demand_Lag_1"]

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_r2 = r2_score(y_test, baseline_predictions)

print("Previous-Month Baseline")
print("MAE :", round(baseline_mae, 4))
print("RMSE:", round(baseline_rmse, 4))
print("R²  :", round(baseline_r2, 4))

Previous-Month Baseline
MAE : 4.5893
RMSE: 5.8858
R²  : 0.4595


In [68]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=10.0)

ridge_model.fit(X_train_processed, y_train)

ridge_predictions = ridge_model.predict(X_test_processed)

print("Ridge model trained successfully.")
print("Number of predictions:", len(ridge_predictions))

Ridge model trained successfully.
Number of predictions: 112


In [69]:
ridge_mae = mean_absolute_error(y_test, ridge_predictions)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_predictions))
ridge_r2 = r2_score(y_test, ridge_predictions)

print("Ridge Regression")
print("MAE :", round(ridge_mae, 4))
print("RMSE:", round(ridge_rmse, 4))
print("R²  :", round(ridge_r2, 4))

Ridge Regression
MAE : 5.9473
RMSE: 7.6823
R²  : 0.0792


In [70]:
# Create demand-change target

y_train_change = (
    train_df["Monthly_Demand"] -
    train_df["Historical_Demand_Lag_1"]
)

y_test_change = (
    test_df["Monthly_Demand"] -
    test_df["Historical_Demand_Lag_1"]
)

print("Demand-change training target:")
print(y_train_change.describe())

print("\nDemand-change testing target:")
print(y_test_change.describe())

Demand-change training target:
count    368.000000
mean       0.086957
std        3.647707
min      -11.000000
25%       -2.000000
50%        0.000000
75%        2.000000
max       11.000000
dtype: float64

Demand-change testing target:
count    112.000000
mean       1.196429
std        5.788836
min      -12.000000
25%       -2.000000
50%        0.000000
75%        5.000000
max       17.000000
dtype: float64


In [71]:
# Train Ridge Regression to predict demand change

ridge_change_model = Ridge(alpha=10.0)

# Remove rows where demand change target is missing
valid_train = y_train_change.notna()

ridge_change_model.fit(
    X_train_processed[valid_train],
    y_train_change[valid_train]
)

predicted_change = ridge_change_model.predict(X_test_processed)

print("Ridge demand-change model trained successfully.")
print("Number of predicted changes:", len(predicted_change))

Ridge demand-change model trained successfully.
Number of predicted changes: 112


In [72]:
# Convert predicted demand change into predicted demand

previous_demand = X_test["Historical_Demand_Lag_1"].values

ridge_change_predictions = previous_demand + predicted_change

print("Number of final demand predictions:", len(ridge_change_predictions))
print("\nFirst 10 predicted demand values:")
print(ridge_change_predictions[:10])

Number of final demand predictions: 112

First 10 predicted demand values:
[5.09435664 8.41346293 4.7614174  5.09991996 7.12535489 9.77807123
 3.45694727 5.55711217 9.28502942 5.02890548]


In [73]:
# Evaluate Ridge demand-change model

ridge_change_mae = mean_absolute_error(y_test, ridge_change_predictions)
ridge_change_rmse = np.sqrt(
    mean_squared_error(y_test, ridge_change_predictions)
)
ridge_change_r2 = r2_score(y_test, ridge_change_predictions)

print("Ridge Regression - Demand Change")
print("MAE :", round(ridge_change_mae, 4))
print("RMSE:", round(ridge_change_rmse, 4))
print("R²  :", round(ridge_change_r2, 4))

Ridge Regression - Demand Change
MAE : 6.0336
RMSE: 7.7439
R²  : 0.0644


In [74]:
# Model comparison

model_results = pd.DataFrame({
    "Model": [
        "Previous-Month Baseline",
        "Ridge Regression",
        "Ridge - Demand Change"
    ],
    "MAE": [
        baseline_mae,
        ridge_mae,
        ridge_change_mae
    ],
    "RMSE": [
        baseline_rmse,
        ridge_rmse,
        ridge_change_rmse
    ],
    "R2": [
        baseline_r2,
        ridge_r2,
        ridge_change_r2
    ]
})

print(model_results.round(4))

                     Model     MAE    RMSE      R2
0  Previous-Month Baseline  4.5893  5.8858  0.4595
1         Ridge Regression  5.9473  7.6823  0.0792
2    Ridge - Demand Change  6.0336  7.7439  0.0644


In [75]:
results_path = "/content/MediRent-Ai/data/processed/demand_model_comparison.csv"

model_results.to_csv(results_path, index=False)

print("Model comparison saved to:", results_path)
print("\nSaved results:")
print(model_results.round(4))

Model comparison saved to: /content/MediRent-Ai/data/processed/demand_model_comparison.csv

Saved results:
                     Model     MAE    RMSE      R2
0  Previous-Month Baseline  4.5893  5.8858  0.4595
1         Ridge Regression  5.9473  7.6823  0.0792
2    Ridge - Demand Change  6.0336  7.7439  0.0644


In [76]:
import os

print("File exists:", os.path.exists(results_path))

if os.path.exists(results_path):
    print("File size:", os.path.getsize(results_path), "bytes")
    print("\nSaved CSV:")
    print(pd.read_csv(results_path))

File exists: True
File size: 250 bytes

Saved CSV:
                     Model       MAE      RMSE        R2
0  Previous-Month Baseline  4.589286  5.885818  0.459513
1         Ridge Regression  5.947332  7.682255  0.079235
2    Ridge - Demand Change  6.033626  7.743887  0.064402


In [ ]:
## 6. Model Selection

The demand prediction models were evaluated using a time-aware train-test split.

- Training period: January 2024 – December 2025
- Testing period: January 2026 – July 2026
- Evaluation metrics: MAE, RMSE, and R²

The Previous-Month Baseline, Ridge Regression, and Ridge Regression with demand-change prediction were evaluated on the 2026 holdout data. Random Forest and XGBoost were also evaluated earlier as candidate models.

Based on the current holdout results, the Previous-Month Baseline achieved the lowest MAE and RMSE and the highest R² among the models evaluated in the current notebook.

Therefore, the Previous-Month Baseline is retained as the demand forecasting approach for the current implementation.

The ML models were not forced to outperform the baseline. Model selection was based on the actual validation results.

In [78]:
!cd /content/MediRent-Ai && git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/demand_model_comparison.csv

nothing added to commit but untracked files present (use "git add" to track)


In [79]:
!cd /content/MediRent-Ai && git add data/processed/demand_model_comparison.csv

In [80]:
!cd /content/MediRent-Ai && git commit -m "Add demand model comparison results"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@53e570e3c103.(none)')


In [81]:
!cd /content/MediRent-Ai && git config user.name "Atha1407" && git config user.email "atharvaingle40@gmail.com"

In [82]:
!cd /content/MediRent-Ai && git status --short

A  data/processed/demand_model_comparison.csv


In [83]:
!ls -lh /content/MediRent-Ai/notebooks/03_Demand_Prediction.ipynb

ls: cannot access '/content/MediRent-Ai/notebooks/03_Demand_Prediction.ipynb': No such file or directory


In [84]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "03_Demand_Prediction.ipynb":
            print(os.path.join(root, file))